In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

# MNIST dataset
root_path = '/home/storopoli/Downloads'


# Pequena transformação para tensores e normalizando o tamanho
trans = transforms.Compose([transforms.Resize((224, 224)),
                            transforms.Grayscale(num_output_channels=3), # resnet espera 3 canais
                            transforms.ToTensor(), 
                            transforms.Normalize((0.1307, 0.1307, 0.1307), (0.3081, 0.3081, 0.3081))])

# Train Dataset
train_dataset = torchvision.datasets.MNIST(root=root_path, train=True, transform=trans, download=True)

# Test Datasets
mnist_dataset = torchvision.datasets.MNIST(root=root_path, train=False, transform=trans)
#imagenet_dataset = torchvision.datasets.ImageNet(root=root_path, train=False, transform=trans)


In [2]:
from torch.utils.data import DataLoader

batch_size = 32

train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)

mnistTest_loader = DataLoader(dataset=mnist_dataset, batch_size=batch_size, shuffle=False)
#imagenetTest_loader = DataLoader(dataset=imagenet_dataset, batch_size=batch_size, shuffle=False)


In [3]:
from torchvision.models import resnet18, ResNet18_Weights

model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)  # Carrega o modelo pré-treinado
model.fc = nn.Linear(model.fc.in_features, 10)  # Ajusta a última camada para 10 classes (MNIST)



In [4]:
from torchvision.models import resnet18, ResNet18_Weights

model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)  # Carrega o modelo pré-treinado

# 3.1 - Congele todas as camadas (Layer 1, 2, 3 e 4) e treine apenas a última camada (fc).
# for param in model.parameters():
#     param.requires_grad = False  # Congela todas as camadas

# #3.2 - Fine tuning parcial “descongelando” apenas o último bloco da rede resnet18 (Layer 4) e última camada (fc)
# for name, param in model.named_parameters():
#     if "layer4" in name or "fc" in name:
#         param.requires_grad = True  # Descongela apenas o Layer 4 e a camada fc
#     else:
#         param.requires_grad = False  # Congela as outras camadas
        
# #3.3 - Fine tuning total “descongelando” todos os blocos da rede resnet18 (Layer 1-4) e última camada (fc)
for param in model.parameters():
    param.requires_grad = True  # Descongela todas as camadas


model.fc = nn.Linear(model.fc.in_features, 10)  # Ajusta a última camada para 10 classes (MNIST)


In [5]:
from torch.optim import Adam

# Hiperparâmetros
loss_fn = nn.CrossEntropyLoss()
learning_rate = 0.001
epochs = 6

# Instânciar o Otimizador Adam
optimizer = Adam(model.parameters(), lr=learning_rate)

In [6]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

True
NVIDIA GeForce RTX 3050 Laptop GPU


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [7]:
# Treinar o Modelo
total_step = len(train_loader) # quantos batches eu tenho

# Listas vazias
loss_list = []
acc_list = []

for epoch in range(epochs):
    for i, (images, labels) in enumerate(train_loader):

        # Move tensores para o dispositivo configurado (CPU ou GPU)
        images, labels = images.to(device), labels.to(device)

        # Gera a propagação (feed forward)
        outputs = model(images)

        # Calcula a função-custo
        loss = loss_fn(outputs, labels)
        loss_list.append(loss.item())

        # Retro-propagação (Backprop) e a otimização com Adam
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Acurácia
        total = labels.size(0)
        _, predicted = torch.max(outputs.data, 1)
        correct = (predicted == labels).sum().item()
        acc_list.append(correct / total)
        if (i + 1) % 100 == 0:
            print(f"Época [{epoch+1}/{epochs}], Step [{i+1}/{total_step}], Custo: {round(loss.item(), 3)}, Acurácia: {round((correct / total) * 100, 3)}")

Época [1/6], Step [100/1875], Custo: 0.549, Acurácia: 84.375
Época [1/6], Step [200/1875], Custo: 0.282, Acurácia: 93.75
Época [1/6], Step [300/1875], Custo: 0.042, Acurácia: 100.0
Época [1/6], Step [400/1875], Custo: 0.074, Acurácia: 96.875
Época [1/6], Step [500/1875], Custo: 0.005, Acurácia: 100.0
Época [1/6], Step [600/1875], Custo: 0.016, Acurácia: 100.0
Época [1/6], Step [700/1875], Custo: 0.119, Acurácia: 96.875
Época [1/6], Step [800/1875], Custo: 0.022, Acurácia: 100.0
Época [1/6], Step [900/1875], Custo: 0.108, Acurácia: 93.75
Época [1/6], Step [1000/1875], Custo: 0.065, Acurácia: 96.875
Época [1/6], Step [1100/1875], Custo: 0.003, Acurácia: 100.0
Época [1/6], Step [1200/1875], Custo: 0.004, Acurácia: 100.0
Época [1/6], Step [1300/1875], Custo: 0.074, Acurácia: 96.875
Época [1/6], Step [1400/1875], Custo: 0.092, Acurácia: 96.875
Época [1/6], Step [1500/1875], Custo: 0.015, Acurácia: 100.0
Época [1/6], Step [1600/1875], Custo: 0.198, Acurácia: 96.875
Época [1/6], Step [1700/18

In [8]:
from torchmetrics.classification import MulticlassF1Score

f1_metric = MulticlassF1Score(num_classes=10, average='macro').to(device)

model.eval() # coloca o modelo em modo de avaliação (sem calcular gradientes)

with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in mnistTest_loader:

        images, labels = images.to(device), labels.to(device)

        # Feed-forward com as imagens de teste
        outputs = model(images)
        
        # gera predições usando a função max()
        _, predicted = torch.max(outputs.data, 1)
        
        # Acumula total e corretas
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        f1_metric(predicted, labels)

    f1 = f1_metric.compute()
    print(f"F1 Score (macro) do Modelo em 10k imagens de teste: {f1:.3f}")
    print(f"Acurácia do Modelo em 10k imagens de teste: {round((correct / total) * 100, 3)}")

F1 Score (macro) do Modelo em 10k imagens de teste: 0.993
Acurácia do Modelo em 10k imagens de teste: 99.29
